# 04 · Monitoreo del pipeline y del job
- Event log del pipeline: progreso de cada flow y métricas de expectations.
- Historial de corridas del job desde `system.lakeflow`.
- Costos (si `system.billing` está disponible).

In [ ]:
dbutils.widgets.text("catalog", "electrocasa_dev", "Catálogo del proyecto")
catalogo = dbutils.widgets.get("catalog")
print("Catálogo:", catalogo)

In [ ]:
sql = """
-- Event log: estado de cada actualización del pipeline (vía cualquiera de sus tablas)
SELECT timestamp, event_type, message
FROM event_log(TABLE(${electrocasa.catalog}.silver.ventas))
WHERE event_type IN ('create_update', 'update_progress')
ORDER BY timestamp DESC
LIMIT 30;
""".replace("${electrocasa.catalog}", catalogo)
resultado = None
for sentencia in [s.strip() for s in sql.split(";")]:
    if sentencia and not all(l.strip().startswith("--") for l in sentencia.splitlines() if l.strip()):
        resultado = spark.sql(sentencia)
if resultado is not None and len(resultado.columns) > 0:
    display(resultado)

In [ ]:
sql = """
-- Métricas de expectations por tabla y regla (filas que pasaron y que fallaron)
SELECT
  timestamp,
  origin.flow_name                                   AS flujo,
  exp.name                                           AS regla,
  exp.passed_records                                 AS pasaron,
  exp.failed_records                                 AS fallaron
FROM event_log(TABLE(${electrocasa.catalog}.silver.ventas)),
LATERAL explode(from_json(details:flow_progress:data_quality:expectations,
        'array<struct<name:string,dataset:string,passed_records:bigint,failed_records:bigint>>')) AS t(exp)
WHERE event_type = 'flow_progress'
  AND details:flow_progress:data_quality:expectations IS NOT NULL
ORDER BY timestamp DESC, flujo, regla;
""".replace("${electrocasa.catalog}", catalogo)
resultado = None
for sentencia in [s.strip() for s in sql.split(";")]:
    if sentencia and not all(l.strip().startswith("--") for l in sentencia.splitlines() if l.strip()):
        resultado = spark.sql(sentencia)
if resultado is not None and len(resultado.columns) > 0:
    display(resultado)

In [ ]:
sql = """
-- Historial de corridas del job (system.lakeflow)
SELECT j.name, r.run_id, r.period_start_time, r.period_end_time,
       timestampdiff(SECOND, r.period_start_time, r.period_end_time) AS segundos,
       r.result_state, r.trigger_type
FROM system.lakeflow.job_run_timeline r
JOIN system.lakeflow.jobs j USING (workspace_id, job_id)
WHERE j.name LIKE '%electrocasa%'
ORDER BY r.period_start_time DESC
LIMIT 20;
""".replace("${electrocasa.catalog}", catalogo)
resultado = None
for sentencia in [s.strip() for s in sql.split(";")]:
    if sentencia and not all(l.strip().startswith("--") for l in sentencia.splitlines() if l.strip()):
        resultado = spark.sql(sentencia)
if resultado is not None and len(resultado.columns) > 0:
    display(resultado)

In [ ]:
sql = """
-- Duración por tarea en las últimas corridas
SELECT t.run_id, t.task_key, t.period_start_time, t.period_end_time, t.result_state
FROM system.lakeflow.job_task_run_timeline t
JOIN system.lakeflow.jobs j USING (workspace_id, job_id)
WHERE j.name LIKE '%electrocasa%'
ORDER BY t.period_start_time DESC
LIMIT 40;
""".replace("${electrocasa.catalog}", catalogo)
resultado = None
for sentencia in [s.strip() for s in sql.split(";")]:
    if sentencia and not all(l.strip().startswith("--") for l in sentencia.splitlines() if l.strip()):
        resultado = spark.sql(sentencia)
if resultado is not None and len(resultado.columns) > 0:
    display(resultado)

In [ ]:
sql = """
-- Costo aproximado por recurso del proyecto (DBUs) en los últimos 30 días
SELECT usage_metadata.job_id, usage_metadata.dlt_pipeline_id, sku_name,
       sum(usage_quantity) AS dbus
FROM system.billing.usage
WHERE usage_date >= current_date() - INTERVAL 30 DAYS
  AND (usage_metadata.job_name LIKE '%electrocasa%' OR usage_metadata.dlt_pipeline_id IS NOT NULL)
GROUP BY ALL
ORDER BY dbus DESC;
""".replace("${electrocasa.catalog}", catalogo)
resultado = None
for sentencia in [s.strip() for s in sql.split(";")]:
    if sentencia and not all(l.strip().startswith("--") for l in sentencia.splitlines() if l.strip()):
        resultado = spark.sql(sentencia)
if resultado is not None and len(resultado.columns) > 0:
    display(resultado)